In [1]:
!python -V

Python 3.9.12


In [ ]:
import pandas as pd
import numpy as np
import pickle
from sklearn.feature_extraction import DictVectorizer
from sklearn.metrics import mean_squared_error

In [ ]:
import mlflow 

mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("nyc-taxi-experiment")

2026/07/21 15:08:57 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/07/21 15:08:57 INFO mlflow.store.db.utils: Updating database tables
INFO  [alembic.runtime.migration] Context impl SQLiteImpl.
INFO  [alembic.runtime.migration] Will assume non-transactional DDL.
INFO  [alembic.runtime.migration] Context impl SQLiteImpl.
INFO  [alembic.runtime.migration] Will assume non-transactional DDL.


<Experiment: artifact_location='/workspaces/mlops-zoomcamp/02-experiment-tracking/mlruns/1', creation_time=1784583723573, experiment_id='1', last_update_time=1784583723573, lifecycle_stage='active', name='nyc-taxi-experiment', tags={}>

In [3]:
def read_dataframe(filename):
    if filename.endswith('.csv'):
        df = pd.read_csv(filename)

        df.lpep_dropoff_datetime = pd.to_datetime(df.lpep_dropoff_datetime)
        df.lpep_pickup_datetime = pd.to_datetime(df.lpep_pickup_datetime)
    elif filename.endswith('.parquet'):
        df = pd.read_parquet(filename)

    df['duration'] = df.lpep_dropoff_datetime - df.lpep_pickup_datetime
    df['duration_t'] = df.duration.apply(lambda td: td.total_seconds() / 60)

    df = df[(df.duration_t >= 0) & (df.duration_t <= 60)]
    
    df = df[df.trip_type.notna()]
    
    df['lpep_pickup_hour']=df['lpep_pickup_datetime'].dt.hour
    
    df['lpep_pickup_dayofweek']=df['lpep_pickup_datetime'].dt.dayofweek
    
    df['lpep_pickup_weekend']=df['lpep_pickup_dayofweek'].apply(lambda x: 'weekend' if x >= 5 else 'weekday')
    
    df['lpep_pickup_weekend_hour'] = df['lpep_pickup_weekend'].astype(str) + '_' + df['lpep_pickup_hour'].astype(str)
    
    df['PU_DO_LocationID'] = df['PULocationID'].astype(str) + '_' + df['DOLocationID'].astype(str)

    categorical = ['lpep_pickup_hour','PU_DO_LocationID']
    df[categorical] = df[categorical].astype(str)
    
    df['congestion_surcharge'].fillna(0, inplace=True)
    df['improvement_surcharge'].fillna(0, inplace=True)
    
    return df

In [5]:
df_train = read_dataframe('data/green_tripdata_2024-01.parquet')
df_val = read_dataframe('data/green_tripdata_2024-02.parquet')

In [14]:
import xgboost as xgb

In [15]:
from hyperopt import fmin, tpe, hp, STATUS_OK, Trials
from hyperopt.pyll import scope

In [16]:
categorical = [#'PU_DO_LocationID',
               'lpep_pickup_weekend_hour', 'trip_type', 'congestion_surcharge', 'improvement_surcharge', 'PULocationID', 'DOLocationID']
numerical = ['trip_distance']

dv = DictVectorizer()

train_dicts = df_train[categorical + numerical].to_dict(orient='records')
X_train = dv.fit_transform(train_dicts)

val_dicts = df_val[categorical + numerical].to_dict(orient='records')
X_val = dv.transform(val_dicts)

with open('models/preprocessor_v2.b', 'wb') as f_out:
    pickle.dump((dv), f_out)
    
target = 'duration_t'
y_train = df_train[target].values
y_val = df_val[target].values

In [17]:
train = xgb.DMatrix(X_train, label=y_train)
valid = xgb.DMatrix(X_val, label=y_val)

In [ ]:
def objective(params):
    with mlflow.start_run():
        mlflow.set_tag("developer", "sameh")
        mlflow.log_param("train-data-path", "data/green_tripdata_2024-01.parquet")
        mlflow.log_param("val-data-path", "data/green_tripdata_2024-02.parquet")
        mlflow.log_param("model_type", "xgboost")
        mlflow.log_params(params)
        mlflow.log_artifact(local_path='models/preprocessor_v2.b', artifact_path='preprocessor')
        booster = xgb.train(
            params=params,
            dtrain=train,
            num_boost_round=100,
            evals=[(valid, 'validation')],
            early_stopping_rounds=5
        )
        y_pred = booster.predict(valid)
        rmse = np.sqrt(mean_squared_error(y_val, y_pred))
        mlflow.log_metric("rmse", rmse)
        mlflow.xgboost.log_model(booster, artifact_path="models_mlflow")

    return {'loss': rmse, 'status': STATUS_OK}

In [20]:
search_space = {
    'max_depth': scope.int(hp.quniform('max_depth', 4, 25, 1)),
    'learning_rate': hp.loguniform('learning_rate', -4, -2),
    'reg_alpha': hp.loguniform('reg_alpha', -5, -1),
    'reg_lambda': hp.loguniform('reg_lambda', -6, -1),
    'min_child_weight': hp.loguniform('min_child_weight', -2, 2),
    #'objective': 'reg:linear',
    'seed': 42
}

best_result = fmin(
    fn=objective,
    space=search_space,
    algo=tpe.suggest,
    max_evals=5,
    trials=Trials()
)

[0]	validation-rmse:8.64748                                                          
[1]	validation-rmse:8.36680                                                          
[2]	validation-rmse:8.10546                                                          
[3]	validation-rmse:7.86215                                                          
[4]	validation-rmse:7.63451                                                          
[5]	validation-rmse:7.42142                                                          
[6]	validation-rmse:7.22355                                                          
[7]	validation-rmse:7.03946                                                          
[8]	validation-rmse:6.86801                                                          
[9]	validation-rmse:6.70777                                                          
[10]	validation-rmse:6.55586                                                         
[11]	validation-rmse:6.41673                          

[95]	validation-rmse:4.77661                                                         
[96]	validation-rmse:4.77651                                                         
[97]	validation-rmse:4.77649                                                         
[98]	validation-rmse:4.77648                                                         
[99]	validation-rmse:4.77628                                                         
[0]	validation-rmse:8.76498                                                          
[1]	validation-rmse:8.59116                                                          
[2]	validation-rmse:8.42528                                                          
[3]	validation-rmse:8.26750                                                          
[4]	validation-rmse:8.11665                                                          
[5]	validation-rmse:7.97243                                                          
[6]	validation-rmse:7.83481                           

[90]	validation-rmse:5.02536                                                         
[91]	validation-rmse:5.02204                                                         
[92]	validation-rmse:5.01934                                                         
[93]	validation-rmse:5.01562                                                         
[94]	validation-rmse:5.00841                                                         
[95]	validation-rmse:5.00538                                                         
[96]	validation-rmse:5.00232                                                         
[97]	validation-rmse:4.99925                                                         
[98]	validation-rmse:4.99620                                                         
[99]	validation-rmse:4.99216                                                         
[0]	validation-rmse:8.40431                                                          
[1]	validation-rmse:7.92699                           

[85]	validation-rmse:4.73174                                                         
[86]	validation-rmse:4.73208                                                         
[87]	validation-rmse:4.73210                                                         
[88]	validation-rmse:4.73220                                                         
[89]	validation-rmse:4.73112                                                         
[90]	validation-rmse:4.72956                                                         
[91]	validation-rmse:4.72960                                                         
[92]	validation-rmse:4.73026                                                         
[93]	validation-rmse:4.73016                                                         
[94]	validation-rmse:4.72990                                                         
[0]	validation-rmse:8.67489                                                          
[1]	validation-rmse:8.41989                           

[85]	validation-rmse:5.05869                                                         
[86]	validation-rmse:5.05776                                                         
[87]	validation-rmse:5.05743                                                         
[88]	validation-rmse:5.05745                                                         
[89]	validation-rmse:5.05710                                                         
[90]	validation-rmse:5.05676                                                         
[91]	validation-rmse:5.05645                                                         
[92]	validation-rmse:5.05711                                                         
[93]	validation-rmse:5.05644                                                         
[94]	validation-rmse:5.05588                                                         
[95]	validation-rmse:5.05591                                                         
[96]	validation-rmse:5.05542                          

[80]	validation-rmse:5.17350                                                         
[81]	validation-rmse:5.16476                                                         
[82]	validation-rmse:5.15449                                                         
[83]	validation-rmse:5.14535                                                         
[84]	validation-rmse:5.13689                                                         
[85]	validation-rmse:5.12694                                                         
[86]	validation-rmse:5.11897                                                         
[87]	validation-rmse:5.11069                                                         
[88]	validation-rmse:5.10321                                                         
[89]	validation-rmse:5.09568                                                         
[90]	validation-rmse:5.08853                                                         
[91]	validation-rmse:5.08167                          

In [ ]:
with mlflow.start_run():

    mlflow.set_tag("developer", "sameh")
    mlflow.log_param("train-data-path", "data/green_tripdata_2024-01.parquet")
    mlflow.log_param("val-data-path", "data/green_tripdata_2024-02.parquet")
    mlflow.log_param("model_type", "xgboost")


    best_params={
        "learning_rate": 0.08944777868968097,
        "max_depth": 15,
        "min_child_weight": 3.9466315707096298,
        #"objective": "reg:linear",
        "reg_alpha": 0.1359070464297258,
        "reg_lambda": 0.0031192521420431406,
        "seed": 42
    }
    
    mlflow.xgboost.autolog(disable=True)
    mlflow.log_params(best_params)

    booster = xgb.train(
        params=best_params,
        dtrain=train,
        num_boost_round=100,
        evals=[(valid, 'validation')],
        early_stopping_rounds=5
    )
    y_pred = booster.predict(valid)
    rmse = np.sqrt(mean_squared_error(y_val, y_pred))
    mlflow.log_metric("rmse", rmse)
    

    mlflow.log_artifact(local_path='models/preprocessor_v2.b', artifact_path='preprocessor')
    
    mlflow.xgboost.log_model(booster, artifact_path="models_mlflow")

[0]	validation-rmse:8.40431
[1]	validation-rmse:7.92699
[2]	validation-rmse:7.50950
[3]	validation-rmse:7.14260
[4]	validation-rmse:6.82050
[5]	validation-rmse:6.54536
[6]	validation-rmse:6.29930
[7]	validation-rmse:6.08549
[8]	validation-rmse:5.90704
[9]	validation-rmse:5.74680
[10]	validation-rmse:5.61302
[11]	validation-rmse:5.50027
[12]	validation-rmse:5.39774
[13]	validation-rmse:5.31390
[14]	validation-rmse:5.23984
[15]	validation-rmse:5.17869
[16]	validation-rmse:5.12690
[17]	validation-rmse:5.08622
[18]	validation-rmse:5.04902
[19]	validation-rmse:5.01583
[20]	validation-rmse:4.98670
[21]	validation-rmse:4.96380
[22]	validation-rmse:4.94382
[23]	validation-rmse:4.92509
[24]	validation-rmse:4.90675
[25]	validation-rmse:4.89307
[26]	validation-rmse:4.88190
[27]	validation-rmse:4.87029
[28]	validation-rmse:4.85731
[29]	validation-rmse:4.84657
[30]	validation-rmse:4.83874
[31]	validation-rmse:4.82777
[32]	validation-rmse:4.82293
[33]	validation-rmse:4.81758
[34]	validation-rmse:4.8

2026/07/21 14:39:11 WARNING mlflow.xgboost: Failed to infer model signature: could not sample data to infer model signature: please ensure that autologging is enabled before constructing the dataset.
2026/07/21 14:39:11 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/07/21 14:39:12 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/home/codespace/anaconda3/lib/python3.9/site-packages/xgboost/core.py:158: UserWarning: [14:39:12] WARNING: /workspace/src/c_api/c_api.cc:1374: Saving model in the UBJSON format as default.  You can use file extension: `json`, `ubj` or `deprecated` to choose between formats."
2026/07/21 14:39:15 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: /tmp/tmp9nu2l2ww/model, flavor: xgboost). Fall back to return ['xgboost==2.1.4']. Set logging level to DEBUG to see the full traceback. 
2026/07/21 14:39:15 WARNING mlflow.models.mod